# 01 数据摸底（EDA）

先在 BigQuery 上把 `dwd_events_clean` 建好，再用这个 notebook 看整体趋势、事件量级和维度分布。
没有 GCP 时把 `OFFLINE = True`，用合成数据跑通代码。

In [ ]:
import os, sys
sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt

OFFLINE = True
PROJECT = os.environ.get('GCP_PROJECT_ID', 'ecom-attribution')
DATASET = os.environ.get('BQ_DATASET', 'ga4_attr')

In [ ]:
if OFFLINE:
    from attribution.synthetic import generate_events, SyntheticConfig
    from attribution.metrics import clean_events, daily_total
    events, stats = clean_events(generate_events(SyntheticConfig(users_per_day=2000)))
    daily = daily_total(events)
    print(stats)
else:
    from google.cloud import bigquery
    client = bigquery.Client(project=PROJECT)
    daily = client.query(f'SELECT * FROM `{PROJECT}.{DATASET}.ads_daily_total` ORDER BY event_dt').to_dataframe()
daily.head()

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(10, 9), sharex=True)
for ax, col in zip(axes, ['gmv', 'uv', 'cr', 'aov']):
    ax.plot(pd.to_datetime(daily['event_dt']), daily[col])
    ax.set_ylabel(col)
plt.tight_layout()

## 周同比与异动点
和 `sql/03_ads/04_ads_daily_anomaly.sql` 同样的逻辑：28 日均值 ± 2 倍标准差。

In [ ]:
d = daily.sort_values('event_dt').copy()
d['gmv_wow'] = d['gmv'] / d['gmv'].shift(7) - 1
d['ma28'] = d['gmv'].shift(1).rolling(28, min_periods=7).mean()
d['sd28'] = d['gmv'].shift(1).rolling(28, min_periods=7).std()
d['z'] = (d['gmv'] - d['ma28']) / d['sd28']
d.reindex(d['z'].abs().sort_values(ascending=False).index).head(10)[['event_dt', 'gmv', 'gmv_wow', 'z']]

## 记录
- 事件数 / 用户数：待实测
- 观察到的异动日期：待实测
- 选为案例的日期及理由：待定